3

Task 3: Multiple Linear Regression from Scratch (CarDekho Dataset)
● Exam Task:
1. Load the CarDekho dataset, select at least two numerical predictors (e.g.,
Present_Price, Kms_Driven) and encode one categorical variable (e.g.,
Fuel_Type)
2. Standardize numerical features ( ) and split into an 80:20 train-test split𝑧 = 𝑥−μ
σ
3. Prepend a bias column of and compute parameter weights1. 0 β
^ = (𝑋𝑇𝑋)−1𝑋𝑇𝑦
using purely NumPy matrix operations (no scikit-learn regression functions)
4. Predict test set car selling prices and compute the Residual Sum of Squares
(RSS) and .𝑅2

In [ ]:
import pandas as pd
import numpy as np

# 1. Load the CarDekho dataset
url = 'https://raw.githubusercontent.com/sahil-sharma-git/Car-Dekho-Price-Prediction/master/car%20data.csv'
try:
    df = pd.read_csv(url)
    print("Dataset loaded successfully!")
except Exception as e:
    print(f"Error loading dataset: {e}")
    # Fallback dummy dataset matching the exact structure if URL is offline
    print("Creating fallback dummy dataset...")
    np.random.seed(42)
    df = pd.DataFrame({
        'Present_Price': np.random.uniform(1.0, 30.0, 301),
        'Kms_Driven': np.random.randint(500, 100000, 301),
        'Fuel_Type': np.random.choice(['Petrol', 'Diesel', 'CNG'], 301),
        'Selling_Price': np.random.uniform(0.5, 25.0, 301)
    })

display(df.head())

Error loading dataset: HTTP Error 404: Not Found
Creating fallback dummy dataset...


,Present_Price,Kms_Driven,Fuel_Type,Selling_Price
0,11.861663,56679,Petrol,12.523345
1,28.570715,32593,Diesel,18.210954
2,22.227824,70178,Diesel,20.611106
3,18.361096,81238,Diesel,18.102203
4,5.524541,40234,Diesel,13.608400


In [ ]:
# Select predictors and target
# We will use Present_Price, Kms_Driven as numerical features and Fuel_Type as a categorical feature.
X_num = df[['Present_Price', 'Kms_Driven']].values
y = df['Selling_Price'].values

# One-hot encode the categorical variable 'Fuel_Type' manually
fuel_encoded = pd.get_dummies(df['Fuel_Type'], drop_first=True).astype(float).values

# Standardize numerical features: z = (x - mean) / std
num_mean = np.mean(X_num, axis=0)
num_std = np.std(X_num, axis=0)
X_num_scaled = (X_num - num_mean) / num_std

# Combine numerical and categorical features
X_combined = np.hstack((X_num_scaled, fuel_encoded))

# Split into 80:20 train-test split manually
np.random.seed(42)  # For reproducibility
indices = np.random.permutation(len(X_combined))
split_idx = int(len(X_combined) * 0.8)

train_idx, test_idx = indices[:split_idx], indices[split_idx:]

X_train, X_test = X_combined[train_idx], X_combined[test_idx]
y_train, y_test = y[train_idx], y[test_idx]

print(f"Train set shape: {X_train.shape}, Test set shape: {X_test.shape}")

Train set shape: (240, 4), Test set shape: (61, 4)


In [ ]:
# Prepend a bias column of 1.0
X_train_bias = np.hstack((np.ones((X_train.shape[0], 1)), X_train))
X_test_bias = np.hstack((np.ones((X_test.shape[0], 1)), X_test))

# Compute parameter weights using purely NumPy matrix operations: beta = (X^T * X)^(-1) * X^T * y
beta = np.linalg.inv(X_train_bias.T @ X_train_bias) @ X_train_bias.T @ y_train

print("Calculated weights (beta):", beta)

Calculated weights (beta): [13.44079436 -0.05184692  0.22766269 -0.75204259  0.90301323]


In [ ]:
# Predict test set car selling prices
y_pred = X_test_bias @ beta

# Compute Residual Sum of Squares (RSS)
rss = np.sum((y_test - y_pred) ** 2)

# Compute R-squared (R2)
tss = np.sum((y_test - np.mean(y_test)) ** 2)
r2 = 1 - (rss / tss)

print(f"Residual Sum of Squares (RSS): {rss:.4f}")
print(f"R-squared (R2) Metric: {r2:.4f}")

Residual Sum of Squares (RSS): 2491.9620
R-squared (R2) Metric: -0.0317
